# Hierarchical Pipeline vs. Traditional Baselines
We compare our 4-Tier Hierarchical Pipeline against the two industry-standard approaches:
- **Baseline A (The Accuracy Brute):** A single heavy model (LightGBM) classifying all 6 activities.
- **Baseline B (The Speed Brute):** A single light model (LinearSVC) classifying all 6 activities.

In [7]:
import numpy as np
import joblib
import time
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

In [8]:
print("Loading data and artifacts...")

# Load full training and test sets
X_train_full = np.load('artifacts/X_train_scaled.npy')
y_train = np.load('artifacts/y_train_encoded.npy')
X_test_full = np.load('artifacts/X_test_scaled.npy')
y_test = np.load('artifacts/y_test_encoded.npy')

# Load selected RF-RFE indices and apply array slicing
indices = joblib.load('artifacts/selected_indices.joblib')
X_train_reduced = X_train_full[:, indices]
X_test_reduced = X_test_full[:, indices]

# Load the Hierarchical Gatekeepers & Specialist (Stage 0 omitted for clean latency testing)
stage1_lr = joblib.load('models/stage1_lr.joblib')
stage1b_svm = joblib.load('models/stage1b_svm.joblib')
stage2_lgbm = joblib.load('models/stage2_lgbm.joblib')

print(f"Data ready. Operating on {len(indices)} selected features out of 561.")

Loading data and artifacts...
Data ready. Operating on 100 selected features out of 561.


In [9]:
print("Training Baseline A: Single LightGBM (Heavy)...")
baseline_lgbm = LGBMClassifier(random_state=42, n_jobs=-1)
baseline_lgbm.fit(X_train_reduced, y_train)

print("Training Baseline B: Single LinearSVC (Light)...")
# dual=False is required for fast execution when samples > features
baseline_svm = LinearSVC(random_state=42, dual=False, max_iter=2000)
baseline_svm.fit(X_train_reduced, y_train)

print("Baselines fully trained and ready for benchmarking.")

Training Baseline A: Single LightGBM (Heavy)...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005831 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25038
[LightGBM] [Info] Number of data points in the train set: 7352, number of used features: 100
[LightGBM] [Info] Start training from score -1.791216
[LightGBM] [Info] Start training from score -1.924514
[LightGBM] [Info] Start training from score -2.009071
[LightGBM] [Info] Start training from score -1.743436
[LightGBM] [Info] Start training from score -1.677246
[LightGBM] [Info] Start training from score -1.653513
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

In [14]:
# ---------------------------------------------------------
# 1. Evaluate Baseline A (Heavy LightGBM)
# ---------------------------------------------------------
start_time = time.perf_counter()
base_lgbm_preds = baseline_lgbm.predict(X_test_reduced)
base_lgbm_time = ((time.perf_counter() - start_time) / len(X_test_reduced)) * 1000
base_lgbm_acc = accuracy_score(y_test, base_lgbm_preds)

# ---------------------------------------------------------
# 2. Evaluate Baseline B (Light LinearSVC)
# ---------------------------------------------------------
start_time = time.perf_counter()
base_svm_preds = baseline_svm.predict(X_test_reduced)
base_svm_time = ((time.perf_counter() - start_time) / len(X_test_reduced)) * 1000
base_svm_acc = accuracy_score(y_test, base_svm_preds)

# ---------------------------------------------------------
# 3. Evaluate Hierarchical Pipeline (SOFT GATING)
# ---------------------------------------------------------
start_time = time.perf_counter()

# THE DIAL: Adjust this to trade battery for accuracy
# 0.85 means "Only bypass LightGBM if we are 85% sure the person is static"
STATIC_CONFIDENCE_THRESHOLD = 0.85 

# 1. Get raw probabilities instead of hard 0/1 predictions
# predict_proba returns an array: [Prob_Static, Prob_Dynamic]
stage1_probs = stage1_lr.predict_proba(X_test_reduced)
prob_static = stage1_probs[:, 0] 

# 2. Create soft routing masks based on the threshold
fast_route_mask = (prob_static >= STATIC_CONFIDENCE_THRESHOLD)
heavy_route_mask = ~fast_route_mask # Everything else goes to the heavy model

hierarchical_preds = np.zeros(len(X_test_reduced), dtype=int)

# 3. Execute Fast Route (Stage 1B LinearSVC)
if np.any(fast_route_mask):
    hierarchical_preds[fast_route_mask] = stage1b_svm.predict(X_test_reduced[fast_route_mask])

# 4. Execute Heavy Route (Using the Baseline LightGBM trained on all 6 classes)
if np.any(heavy_route_mask):
    hierarchical_preds[heavy_route_mask] = baseline_lgbm.predict(X_test_reduced[heavy_route_mask])

end_time = time.perf_counter()

hierarchical_acc = accuracy_score(y_test, hierarchical_preds)
hierarchical_avg_time = ((end_time - start_time) / len(X_test_reduced)) * 1_000_000

# ---------------------------------------------------------
# PRINT THE FINAL DEFENSE
# ---------------------------------------------------------
print("==========================================================")
print(" SOFT GATING SHOWDOWN: ACCURACY vs. LATENCY")
print("==========================================================")
print(f"1. Single Light Model (LinearSVC) : Acc {base_svm_acc*100:.2f}% | Latency {base_svm_time * 1000:.2f} μs")
print(f"2. Single Heavy Model (LightGBM)  : Acc {base_lgbm_acc*100:.2f}% | Latency {base_lgbm_time * 1000:.2f} μs")
print(f"3. Soft Gated Hierarchy (Thr={STATIC_CONFIDENCE_THRESHOLD}) : Acc {hierarchical_acc*100:.2f}% | Avg Latency {hierarchical_avg_time:.2f} μs")
print("==========================================================\n")

# Calculate the actual routing percentages to prove what happened
total_samples = len(X_test_reduced)
bypassed = np.sum(fast_route_mask)
print(f"--- ROUTING MECHANICS ---")
print(f"Data safely routed to Fast SVM : {(bypassed/total_samples)*100:.1f}%")
print(f"Data requiring Heavy LightGBM  : {((total_samples-bypassed)/total_samples)*100:.1f}%\n")

print("--- THE ENGINEERING CONCLUSION ---")
accuracy_retained = (hierarchical_acc / base_lgbm_acc) * 100
latency_saved = ((base_lgbm_time * 1000 - hierarchical_avg_time) / (base_lgbm_time * 1000)) * 100

print(f"By requiring an {STATIC_CONFIDENCE_THRESHOLD*100}% confidence threshold for static routing,")
print(f"the pipeline retains {accuracy_retained:.1f}% of the LightGBM's accuracy")
print(f"while simultaneously reducing average processor active time by {latency_saved:.1f}%.")

 SOFT GATING SHOWDOWN: ACCURACY vs. LATENCY
1. Single Light Model (LinearSVC) : Acc 94.74% | Latency 0.51 μs
2. Single Heavy Model (LightGBM)  : Acc 92.67% | Latency 8.04 μs
3. Soft Gated Hierarchy (Thr=0.85) : Acc 93.42% | Avg Latency 5.78 μs

--- ROUTING MECHANICS ---
Data safely routed to Fast SVM : 52.9%
Data requiring Heavy LightGBM  : 47.1%

--- THE ENGINEERING CONCLUSION ---
By requiring an 85.0% confidence threshold for static routing,
the pipeline retains 100.8% of the LightGBM's accuracy
while simultaneously reducing average processor active time by 28.1%.
